In [1]:
# Install the transformers library if you haven't already
!pip install transformers datasets accelerate

from transformers import AutoModelForCTC, AutoProcessor

# Define the path to your saved model on Google Drive
model_path = '/content/drive/MyDrive/nepali-asr-xlsr300m/4rd-model_checkpoint'

# Load the processor (tokenizer and feature extractor)
processor = AutoProcessor.from_pretrained(model_path)

# Load the fine-tuned model
model = AutoModelForCTC.from_pretrained(model_path)

print("Model and processor loaded successfully!")

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Model and processor loaded successfully!


In [2]:
import soundfile as sf
import torch

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# Define the path to your raw audio file (e.g., a WAV file)
# You can upload an audio file to Colab's file system or use one from Drive.
# For example, if you upload 'my_audio.wav' directly to /content/:
# audio_file_path = '/content/my_audio.wav'
# Or if it's on Google Drive:
audio_file_path = '/content/generated-audio.mp3' # <--- IMPORTANT: Update this path!

# Helper function for audio preprocessing
def preprocess_audio_for_asr(speech_array, target_peak_amplitude=0.9):
    # Convert to mono if stereo
    if speech_array.ndim > 1:
        speech_array = speech_array.mean(axis=1)

    # Normalize peak amplitude
    max_amplitude = np.max(np.abs(speech_array))
    if max_amplitude > 0:
        speech_array = speech_array / max_amplitude * target_peak_amplitude

    return speech_array

# Make sure to import numpy for array operations
import numpy as np
# Import librosa for resampling
import librosa

# 1. Load the audio file
try:
    speech, original_sample_rate = sf.read(audio_file_path)
    print(f"Audio file loaded: {audio_file_path}")
    print(f"Original sample rate: {original_sample_rate} Hz")
    print(f"Audio duration: {len(speech) / original_sample_rate:.2f} seconds")
except Exception as e:
    print(f"Error loading audio file: {e}")
    print("Please ensure the audio_file_path is correct and the file exists.")
    print("You might need to upload an audio file or update the path.")
    raise # Re-raise the exception to stop execution if file is not found

# 1.5. Preprocess audio for better consistency (mono and normalized volume)
speech = preprocess_audio_for_asr(speech) # Normalization doesn't depend on sample rate

# 1.6. Resample audio to the model's expected sampling rate (16000 Hz)
target_sample_rate = processor.feature_extractor.sampling_rate
if original_sample_rate != target_sample_rate:
    print(f"Resampling audio from {original_sample_rate} Hz to {target_sample_rate} Hz...")
    speech = librosa.resample(y=speech.astype(float), orig_sr=original_sample_rate, target_sr=target_sample_rate)
    sample_rate = target_sample_rate # Update sample_rate variable for processor
else:
    sample_rate = original_sample_rate # Keep original if already matching

# 2. Preprocess the audio for the model
# The processor will convert to a PyTorch tensor after confirming the sample rate
input_values = processor(speech, sampling_rate=sample_rate, return_tensors="pt").input_values

# 3. Perform inference with the fine-tuned model
# Make sure the model is in evaluation mode
model.eval()
with torch.no_grad():
    logits = model(input_values).logits

# 4. Decode the logits to get the predicted transcription
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)

# 5. Print the transcription
print("\n--- Transcription Result ---")
print(transcription[0])
print("--------------------------")

Audio file loaded: /content/generated-audio.mp3
Original sample rate: 24000 Hz
Audio duration: 40.66 seconds
Resampling audio from 24000 Hz to 16000 Hz...

--- Transcription Result ---
नेपाल हिमालहरूको देशवो जहाँ कृथछ्वीको सबैबन्दा अग्लो शिखर सगरमाथा अवस्तिक छयहाँको संस्कृतिक विवितथा अ्त्यन्तै लवोचक छ जहाँ विभिन्र जाकजातिन वाषा र धर्मका मानिसहरु सान्ति पूर्वक वस्वबास गर्छन्मालका दृश्य हरीयआलिख्वववड गास्य मैदानहरु र नदीहरू पर्यठकहरूका लागि आक्षप गन्तव्य हुन् नेपाली चनजीवन प्राय कृषीमा आगारिक छ र यहाँका पर्व र चाडहरू समुदाएको एकता र परम्पदालाई प्रतिबिमवित गर्छन्आदुनिक सहरहरूको विकास बहिरहे पनि ग्रामिम जीबनले आफ्नो पारम परिक सैली र सरलताकायम लाखेको छ।
--------------------------


In [5]:
from IPython.display import Audio

# Play the processed audio
Audio(speech, rate=sample_rate)

In [6]:

transcription[0]


'नेपाल हिमालहरूको देशवो जहाँ कृथछ्वीको सबैबन्दा अग्लो शिखर सगरमाथा अवस्तिक छयहाँको संस्कृतिक विवितथा अ्त्यन्तै लवोचक छ जहाँ विभिन्र जाकजातिन वाषा र धर्मका मानिसहरु सान्ति पूर्वक वस्वबास गर्छन्मालका दृश्य हरीयआलिख्वववड गास्य मैदानहरु र नदीहरू पर्यठकहरूका लागि आक्षप गन्तव्य हुन् नेपाली चनजीवन प्राय कृषीमा आगारिक छ र यहाँका पर्व र चाडहरू समुदाएको एकता र परम्पदालाई प्रतिबिमवित गर्छन्आदुनिक सहरहरूको विकास बहिरहे पनि ग्रामिम जीबनले आफ्नो पारम परिक सैली र सरलताकायम लाखेको छ।'

In [7]:
kriyapad_file_path = '/content/kriyapad.txt' # <--- IMPORTANT: Update this path to your actual file!

kriyapad_set = set()
try:
    with open(kriyapad_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            kriyapad_set.add(line.strip())
    print(f"Loaded {len(kriyapad_set)} kriyapad (verbs) into the set.")
except FileNotFoundError:
    print(f"Error: Kriyapad file not found at {kriyapad_file_path}")
    print("Please ensure the path is correct and the file exists.")
except Exception as e:
    print(f"An error occurred while loading kriyapad file: {e}")

Loaded 2398 kriyapad (verbs) into the set.


In [8]:
raw_text = transcription[0]
print(f"Raw text extracted successfully:\n{raw_text[:100]}...")

Raw text extracted successfully:
नेपाल हिमालहरूको देशवो जहाँ कृथछ्वीको सबैबन्दा अग्लो शिखर सगरमाथा अवस्तिक छयहाँको संस्कृतिक विवितथा ...


In [9]:
problematic_suffix_chars = {'ो', 'े', 'ी', 'ु', 'ू', 'ै', 'ा', 'ि'}
print("Problematic suffix characters set created:", problematic_suffix_chars)

Problematic suffix characters set created: {'ै', 'ू', 'ी', 'ु', 'ो', 'े', 'ि', 'ा'}


In [10]:
def punctuate_with_verbs_refined(raw_text, kriyapad_set, problematic_suffixes, min_words=3):
    """
    Optimized version: Tokenizes by word boundaries.
    If back-to-back verbs are found, it prefers placing the purnabiram
    after the last verb in the sequence.
    """
    k_set = set(kriyapad_set)
    words = raw_text.split()
    punctuated_parts = []
    words_since_last_punc = 0

    i = 0
    while i < len(words):
        word = words[i]
        words_since_last_punc += 1

        # Helper to check if a word contains a verb
        def get_verb_info(w):
            if w in k_set: return True, w, ""
            for length in range(len(w), 2, -1):
                prefix = w[:length]
                if prefix in k_set:
                    suffix = w[length:]
                    if not (suffix and suffix[0] in problematic_suffixes):
                        return True, prefix, suffix
            return False, None, None

        is_v, pref, suff = get_verb_info(word)

        if is_v:
            # Look ahead: is the next word also a verb?
            next_is_verb = False
            if i + 1 < len(words):
                next_is_v, _, _ = get_verb_info(words[i+1])
                if next_is_v: next_is_verb = True

            # Put purnabiram if:
            # 1. It's a verb AND
            # 2. Not followed immediately by another verb (prefer latter) AND
            # 3. Minimum word count met
            if not next_is_verb and words_since_last_punc >= min_words:
                if suff == "":
                    punctuated_parts.append(word + "।")
                else:
                    punctuated_parts.append(pref + "।" + suff)
                words_since_last_punc = 0
            else:
                punctuated_parts.append(word)
        else:
            punctuated_parts.append(word)

        i += 1

    final_text = " ".join(punctuated_parts).replace(" ।", "।").replace("। ", "। ")
    final_text = final_text.rstrip("।") + "।"
    return final_text

In [11]:
# Re-apply with 'prefer latter verb' logic
if 'raw_text' in locals() and 'kriyapad_set' in locals():
    refined_punctuated_result = punctuate_with_verbs_refined(raw_text, kriyapad_set, problematic_suffix_chars, min_words=3)
    print("\n--- Optimized Transcription Result (Prefer Latter Verb) ---")
    print(refined_punctuated_result)
    print("----------------------------------------")


--- Optimized Transcription Result (Prefer Latter Verb) ---
नेपाल हिमालहरूको देशवो जहाँ कृथछ्वीको सबैबन्दा अग्लो शिखर सगरमाथा अवस्तिक छयहाँको संस्कृतिक विवितथा अ्त्यन्तै लवोचक छ। जहाँ विभिन्र जाकजातिन वाषा र धर्मका मानिसहरु सान्ति पूर्वक वस्वबास गर्छन्।मालका दृश्य हरीयआलिख्वववड गास्य मैदानहरु र नदीहरू पर्यठकहरूका लागि आक्षप गन्तव्य हुन्। नेपाली चनजीवन प्राय कृषीमा आगारिक छ। र यहाँका पर्व र चाडहरू समुदाएको एकता र परम्पदालाई प्रतिबिमवित गर्छन्।आदुनिक सहरहरूको विकास बहिरहे पनि ग्रामिम जीबनले आफ्नो पारम परिक सैली र सरलताकायम लाखेको छ।
----------------------------------------


In [12]:
print("\n--- Transcription Result ---")
refined_punctuated_result


--- Transcription Result ---


'नेपाल हिमालहरूको देशवो जहाँ कृथछ्वीको सबैबन्दा अग्लो शिखर सगरमाथा अवस्तिक छयहाँको संस्कृतिक विवितथा अ्त्यन्तै लवोचक छ। जहाँ विभिन्र जाकजातिन वाषा र धर्मका मानिसहरु सान्ति पूर्वक वस्वबास गर्छन्।मालका दृश्य हरीयआलिख्वववड गास्य मैदानहरु र नदीहरू पर्यठकहरूका लागि आक्षप गन्तव्य हुन्। नेपाली चनजीवन प्राय कृषीमा आगारिक छ। र यहाँका पर्व र चाडहरू समुदाएको एकता र परम्पदालाई प्रतिबिमवित गर्छन्।आदुनिक सहरहरूको विकास बहिरहे पनि ग्रामिम जीबनले आफ्नो पारम परिक सैली र सरलताकायम लाखेको छ।'

In [13]:
# Install the jiwer library for WER and CER calculation
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 16.5 MB/s eta 0:00:00


In [14]:
import jiwer

# Define the reference (ground truth) text and hypothesis (ASR output) text
# For demonstration, we'll use the previously generated transcription as the hypothesis.
# You should replace 'reference_text' with your actual ground truth.

reference_text = "नेपाल हिमालहरूको देश हो, जहाँ पृथ्वीको सबैभन्दा अग्लो शिखर सगरमाथा अवस्थित छ। यहाँको सांस्कृतिक विविधता अत्यन्तै रोचक छ, जहाँ विभिन्न जातजाति, भाषा र धर्मका मानिसहरू शान्तिपूर्वक बसोबास गर्छन्। हिमालका दृश्य, हरियाली-covered घाँसे मैदानहरू, र नदीहरू पर्यटकहरूका लागि आकर्षक गन्तव्य हुन्। नेपाली जनजीवन प्रायः कृषिमा आधारित छ, र यहाँका पर्व र चाडहरू समुदायको एकता र परम्परालाई प्रतिबिम्बित गर्छन्। आधुनिक शहरहरूको विकास भइरहे पनि, ग्रामीण जीवनले आफ्नो पारम्परिक शैली र सरलता कायम राखेको छ।"
hypothesis_text = "नेपाल हिमालहरूको देशवो जहाँ कृथछ्वीको सबैबन्दा अग्लो शिखर सगरमाथा अवस्तिक छयहाँको संस्कृतिक विवितथा अ्त्यन्तै लवोचक छ। जहाँ विभिन्र जाकजातिन वाषा र धर्मका मानिसहरु सान्ति पूर्वक वस्वबास गर्छन्।मालका दृश्य हरीयआलिख्वववड गास्य मैदानहरु र नदीहरू पर्यठकहरूका लागि आक्षप गन्तव्य हुन्। नेपाली चनजीवन प्राय कृषीमा आगारिक छ। र यहाँका पर्व र चाडहरू समुदाएको एकता र परम्पदालाई प्रतिबिमवित गर्छन्।आदुनिक सहरहरूको विकास बहिरहे पनि ग्रामिम जीबनले आफ्नो पारम परिक सैली र सरलताकायम लाखेको छ।"
print("Reference Text:", reference_text[:100], "...")
print("Predicted Text:", hypothesis_text[:100], "...")
# Calculate WER
wer = jiwer.wer(reference_text, hypothesis_text)

# Calculate CER
cer = jiwer.cer(reference_text, hypothesis_text)

print(f"\nWord Error Rate (WER): {wer:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")

Reference Text: नेपाल हिमालहरूको देश हो, जहाँ पृथ्वीको सबैभन्दा अग्लो शिखर सगरमाथा अवस्थित छ। यहाँको सांस्कृतिक विवि ...
Predicted Text: नेपाल हिमालहरूको देशवो जहाँ कृथछ्वीको सबैबन्दा अग्लो शिखर सगरमाथा अवस्तिक छयहाँको संस्कृतिक विवितथा  ...

Word Error Rate (WER): 0.6528
Character Error Rate (CER): 0.1537
